# AgriCrossGraph — Kaggle runner
Settings (right panel): **Accelerator = GPU T4 x2 or P100**, **Internet = On**. Add your datasets with **+ Add Input**.

In [ ]:
!nvidia-smi

## 1. Get the code
Public repo: just clone. Private repo: add a Kaggle Secret named `GITHUB_TOKEN` (Add-ons → Secrets) and use the second form.

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO = "agricrossgraph"

import os
if os.path.exists(f"/kaggle/working/{REPO}"):
    !cd /kaggle/working/{REPO} && git pull
else:
    !git clone https://github.com/{GITHUB_USER}/{REPO}.git /kaggle/working/{REPO}
    # private repo instead:
    # from kaggle_secrets import UserSecretsClient
    # tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    # !git clone https://{tok}@github.com/{GITHUB_USER}/{REPO}.git /kaggle/working/{REPO}
%cd /kaggle/working/{REPO}
!pip install -q -r requirements.txt

## 2. Find your dataset folders
Copy the exact folder path of Agri-Vision4 from this output into `configs/paths/kaggle.yaml` (`agrivision4_root`), or set it below.

In [ ]:
!python scripts/inspect_dataset.py /kaggle/input --depth 4

In [ ]:
AV4 = "/kaggle/input/agri-vision4"   # <- EDIT to the folder that contains Original_Images / Augmented_Images
P = f"paths.agrivision4_root={AV4}"

## 3. Index the data (manifest) and check counts against the paper
Fix anything listed under **PROBLEMS TO FIX** by adding aliases in `metadata/agrivision4_classes.csv`.

In [ ]:
!python scripts/build_manifest.py --config configs/data/agrivision4.yaml {P}

## 4. Freeze the splits (first time only)
After this, download `splits/` (or `git push` it) so every later run uses the same splits.

In [ ]:
!python scripts/make_splits.py --config configs/experiments/baseline_cnn_loco.yaml {P}
!python scripts/make_splits.py --config configs/experiments/baseline_cnn_within_crop.yaml {P}

## 5. Smoke test (a few minutes)

In [ ]:
!python scripts/train.py --config configs/experiments/smoke_test.yaml {P}

## 6. Real experiments
For long runs use **Save Version → Save & Run All (Commit)**: it runs in the background (up to 12 h) and keeps `/kaggle/working` as the notebook's output.

In [ ]:
!python scripts/train.py --config configs/experiments/baseline_cnn_loco.yaml {P}

In [ ]:
!python scripts/train.py --config configs/experiments/baseline_cnn_within_crop.yaml {P}

## 7. Results table + download

In [ ]:
!python scripts/summarize.py /kaggle/working/outputs
!cd /kaggle/working && zip -qr outputs_no_ckpt.zip outputs -x '*.pt' && zip -qr splits.zip agricrossgraph/splits
print('Download outputs_no_ckpt.zip and splits.zip from the Output panel')